# Shared Training Utilities

This notebook contains reusable functions/classes for all experiment notebooks. It is meant to be loaded with `%run`, not read as the main analysis notebook.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")


def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root by walking upward until config/genes.json is found.
    This works whether the notebook is launched from repo root or notebooks/.
    """
    if "TCGA_TAA_PROJECT_ROOT" in os.environ:
        return Path(os.environ["TCGA_TAA_PROJECT_ROOT"]).expanduser().resolve()

    if start is None:
        start = Path.cwd()

    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "config" / "genes.json").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root. Start Jupyter from the repository root, "
        "or set TCGA_TAA_PROJECT_ROOT=/path/to/repo."
    )


PROJECT_ROOT = find_project_root()
CONFIG_DIR = PROJECT_ROOT / "config"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

for folder in [
    PROCESSED_DATA_DIR,
    RESULTS_DIR / "metrics",
    RESULTS_DIR / "predictions",
    RESULTS_DIR / "histories",
    RESULTS_DIR / "configs",
    RESULTS_DIR / "models",
    REPORTS_DIR,
    FIGURES_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


def load_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


GENE_CONFIG = load_json(CONFIG_DIR / "genes.json")
EXPERIMENT_CONFIG = load_json(CONFIG_DIR / "experiments.json")

TARGET_GENES = GENE_CONFIG["taa_target_genes"]
TARGET_COLS = [f"{g}_rna_log2tpm" for g in TARGET_GENES]

MISSING_TOKENS = [
    "'--", "--", "", " ", "NA", "N/A", "NaN", "nan",
    "Not Reported", "not reported", "Unknown", "unknown",
    "Not Available", "not available", "None", "none",
]


def normalize_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.replace(MISSING_TOKENS, np.nan)
    out = out.replace(r"^\s*'?-?-\s*$", np.nan, regex=True)
    return out


def load_modeling_table() -> pd.DataFrame:
    path = PROCESSED_DATA_DIR / "modeling_table.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run 01_Data_Processing_Pipeline.ipynb first, "
            "or place modeling_table.csv in data/processed/."
        )
    return pd.read_csv(path)


def get_feature_columns(df: pd.DataFrame) -> list[str]:
    id_like = {
        "case_id", "case_barcode", "merge_barcode", "clinical_case_uuid",
        "case_submitter_id", "sample_id", "file_id", "source_file"
    }
    exclude = set(TARGET_COLS) | {c for c in id_like if c in df.columns}
    feature_cols = [c for c in df.columns if c not in exclude]

    # Drop columns that are almost completely empty.
    missing_frac = df[feature_cols].isna().mean()
    feature_cols = [c for c in feature_cols if missing_frac[c] <= 0.70]
    return feature_cols


def split_feature_modalities(feature_cols: list[str]) -> dict[str, list[str]]:
    cn_cols = [c for c in feature_cols if c.endswith("_cn") and c != "project_id_cn"]
    mut_cols = [c for c in feature_cols if c.endswith("_mut") and c != "project_id_mut"]
    clinical_cols = [c for c in feature_cols if c not in set(cn_cols + mut_cols)]
    return {"clinical": clinical_cols, "mutation": mut_cols, "copy_number": cn_cols}


def stratify_if_valid(df: pd.DataFrame, col: str = "project_id"):
    if col not in df.columns:
        return None
    counts = df[col].value_counts()
    if df[col].nunique() <= 1:
        return None
    if counts.min() < 2:
        return None
    return df[col]


def make_splits(df: pd.DataFrame, test_size=0.15, val_size=0.15, seed=42):
    stratify = stratify_if_valid(df, "project_id")
    train_val, test = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=stratify,
    )

    rel_val_size = val_size / (1 - test_size)
    stratify_tv = stratify_if_valid(train_val, "project_id")
    train, val = train_test_split(
        train_val,
        test_size=rel_val_size,
        random_state=seed,
        stratify=stratify_tv,
    )
    return train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def infer_numeric_and_categorical(train_df: pd.DataFrame, feature_cols: list[str]):
    X = normalize_missing_values(train_df[feature_cols].copy())
    numeric_cols = []
    categorical_cols = []

    for col in feature_cols:
        converted = pd.to_numeric(X[col], errors="coerce")
        non_missing = X[col].notna().sum()
        numeric_count = converted.notna().sum()
        numeric_fraction = numeric_count / max(non_missing, 1)

        if numeric_count > 0 and numeric_fraction > 0.80:
            numeric_cols.append(col)
        else:
            categorical_cols.append(col)

    return numeric_cols, categorical_cols


def build_preprocessor(train_df: pd.DataFrame, feature_cols: list[str]):
    numeric_cols, categorical_cols = infer_numeric_and_categorical(train_df, feature_cols)

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ])

    return preprocessor, numeric_cols, categorical_cols


def prepare_feature_frame(df: pd.DataFrame, numeric_cols: list[str], categorical_cols: list[str]) -> pd.DataFrame:
    cols = numeric_cols + categorical_cols
    X = normalize_missing_values(df[cols].copy())

    for col in numeric_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")

    for col in categorical_cols:
        X[col] = X[col].astype("object")

    return X


def evaluate_predictions(y_true: np.ndarray, y_pred: np.ndarray, target_cols: list[str]) -> pd.DataFrame:
    rows = []
    for i, target in enumerate(target_cols):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        rows.append({
            "target": target,
            "r2": r2_score(yt, yp),
            "mae": mean_absolute_error(yt, yp),
            "rmse": float(np.sqrt(mean_squared_error(yt, yp))),
        })
    return pd.DataFrame(rows)


class StandardMLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dims=(256, 128, 64), dropout=0.20, batch_norm=True):
        super().__init__()
        layers = []
        prev = input_dim

        for hidden in hidden_dims:
            layers.append(nn.Linear(prev, hidden))
            if batch_norm:
                layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = hidden

        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class ResidualBlock(nn.Module):
    def __init__(self, hidden_dim: int, dropout=0.20):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
        )
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(x + self.block(x))


class ResidualMLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim=128, num_blocks=3, dropout=0.20):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(*[
            ResidualBlock(hidden_dim=hidden_dim, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        return self.output_layer(x)


class MultiBranchMultiTaskNet(nn.Module):
    def __init__(
        self,
        clinical_dim: int,
        mutation_dim: int,
        copy_number_dim: int,
        output_dim: int,
        branch_hidden_dim=64,
        fusion_hidden_dim=128,
        dropout=0.20,
    ):
        super().__init__()

        def make_branch(input_dim):
            if input_dim == 0:
                return None
            return nn.Sequential(
                nn.Linear(input_dim, branch_hidden_dim),
                nn.BatchNorm1d(branch_hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(branch_hidden_dim, branch_hidden_dim),
                nn.BatchNorm1d(branch_hidden_dim),
                nn.ReLU(),
            )

        self.clinical_branch = make_branch(clinical_dim)
        self.mutation_branch = make_branch(mutation_dim)
        self.copy_number_branch = make_branch(copy_number_dim)

        fusion_input_dim = branch_hidden_dim * sum(dim > 0 for dim in [clinical_dim, mutation_dim, copy_number_dim])

        self.shared_trunk = nn.Sequential(
            nn.Linear(fusion_input_dim, fusion_hidden_dim),
            nn.BatchNorm1d(fusion_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            ResidualBlock(fusion_hidden_dim, dropout=dropout),
            ResidualBlock(fusion_hidden_dim, dropout=dropout),
        )

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_hidden_dim, 64),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(64, 1),
            )
            for _ in range(output_dim)
        ])

    def forward(self, x_clinical, x_mutation, x_copy_number):
        reps = []
        if self.clinical_branch is not None:
            reps.append(self.clinical_branch(x_clinical))
        if self.mutation_branch is not None:
            reps.append(self.mutation_branch(x_mutation))
        if self.copy_number_branch is not None:
            reps.append(self.copy_number_branch(x_copy_number))

        x = torch.cat(reps, dim=1)
        shared = self.shared_trunk(x)
        outputs = [head(shared) for head in self.heads]
        return torch.cat(outputs, dim=1)


def train_torch_model(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    epochs=120,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=20,
    seed=42,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

    val_x = torch.tensor(X_val, dtype=torch.float32).to(device)
    val_y = torch.tensor(y_val, dtype=torch.float32).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    patience_left = patience
    history = []

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_pred = model(val_x)
            val_loss = loss_fn(val_pred, val_y).item()

        train_loss = float(np.mean(batch_losses))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

        if epoch == 1 or epoch % 10 == 0:
            print(f"epoch {epoch:03d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch}. Best val_loss={best_val:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - start_time
    return model, pd.DataFrame(history), best_val, device, elapsed


def train_multibranch_model(
    model,
    train_arrays,
    val_arrays,
    epochs=120,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=20,
    seed=42,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    Xc_train, Xm_train, Xn_train, y_train = train_arrays
    Xc_val, Xm_val, Xn_val, y_val = val_arrays

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_ds = TensorDataset(
        torch.tensor(Xc_train, dtype=torch.float32),
        torch.tensor(Xm_train, dtype=torch.float32),
        torch.tensor(Xn_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

    Xc_val = torch.tensor(Xc_val, dtype=torch.float32).to(device)
    Xm_val = torch.tensor(Xm_val, dtype=torch.float32).to(device)
    Xn_val = torch.tensor(Xn_val, dtype=torch.float32).to(device)
    y_val = torch.tensor(y_val, dtype=torch.float32).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_val = float("inf")
    best_state = None
    patience_left = patience
    history = []

    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []

        for xcb, xmb, xnb, yb in train_loader:
            xcb = xcb.to(device)
            xmb = xmb.to(device)
            xnb = xnb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            pred = model(xcb, xmb, xnb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_pred = model(Xc_val, Xm_val, Xn_val)
            val_loss = loss_fn(val_pred, y_val).item()

        train_loss = float(np.mean(batch_losses))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

        if epoch == 1 or epoch % 10 == 0:
            print(f"epoch {epoch:03d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"Early stopping at epoch {epoch}. Best val_loss={best_val:.4f}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - start_time
    return model, pd.DataFrame(history), best_val, device, elapsed


def save_experiment_outputs(
    experiment_name: str,
    metrics: pd.DataFrame,
    predictions: pd.DataFrame,
    history: pd.DataFrame,
    config: dict,
    model=None,
):
    metrics = metrics.copy()
    history = history.copy()

    metrics["experiment_name"] = experiment_name
    history["experiment_name"] = experiment_name

    metrics_path = RESULTS_DIR / "metrics" / f"{experiment_name}_metrics.csv"
    predictions_path = RESULTS_DIR / "predictions" / f"{experiment_name}_predictions.csv"
    history_path = RESULTS_DIR / "histories" / f"{experiment_name}_history.csv"
    config_path = RESULTS_DIR / "configs" / f"{experiment_name}_config.json"

    metrics.to_csv(metrics_path, index=False)
    predictions.to_csv(predictions_path, index=False)
    history.to_csv(history_path, index=False)

    with open(config_path, "w") as f:
        json.dump(config, f, indent=2, default=str)

    if model is not None:
        model_path = RESULTS_DIR / "models" / f"{experiment_name}.pt"
        torch.save(model.state_dict(), model_path)

    print(f"Saved metrics:     {metrics_path}")
    print(f"Saved predictions: {predictions_path}")
    print(f"Saved history:     {history_path}")
    print(f"Saved config:      {config_path}")


def run_mlp_experiment(config: dict, seed=42, save_model=True):
    experiment_name = config["experiment_name"]
    print(f"\nRunning experiment: {experiment_name}")

    df = load_modeling_table()
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)
    feature_cols = get_feature_columns(df)

    train_df, val_df, test_df = make_splits(df, seed=seed)

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(train_df, feature_cols)

    X_train = preprocessor.fit_transform(prepare_feature_frame(train_df, numeric_cols, categorical_cols))
    X_val = preprocessor.transform(prepare_feature_frame(val_df, numeric_cols, categorical_cols))
    X_test = preprocessor.transform(prepare_feature_frame(test_df, numeric_cols, categorical_cols))

    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(train_df[TARGET_COLS].values)
    y_val = y_scaler.transform(val_df[TARGET_COLS].values)
    y_test_orig = test_df[TARGET_COLS].values

    model = StandardMLP(
        input_dim=X_train.shape[1],
        output_dim=len(TARGET_COLS),
        hidden_dims=tuple(config.get("hidden_dims", [256, 128, 64])),
        dropout=config.get("dropout", 0.20),
        batch_norm=config.get("batch_norm", True),
    )

    model, history, best_val, device, elapsed = train_torch_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        epochs=config.get("epochs", 120),
        batch_size=config.get("batch_size", 64),
        learning_rate=config.get("learning_rate", 1e-3),
        weight_decay=config.get("weight_decay", 1e-4),
        patience=config.get("patience", 20),
        seed=seed,
    )

    model.eval()
    with torch.no_grad():
        pred_scaled = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

    pred = y_scaler.inverse_transform(pred_scaled)
    metrics = evaluate_predictions(y_test_orig, pred, TARGET_COLS)
    metrics["model_family"] = config.get("model_family", "standard_mlp")
    metrics["best_val_loss"] = best_val
    metrics["elapsed_seconds"] = elapsed

    prediction_out = test_df[["case_id", "project_id"] if "project_id" in test_df.columns else ["case_id"]].copy()
    for i, target in enumerate(TARGET_COLS):
        prediction_out[f"{target}_true"] = y_test_orig[:, i]
        prediction_out[f"{target}_pred"] = pred[:, i]

    config_out = dict(config)
    config_out.update({
        "n_rows": len(df),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        "n_raw_features": len(feature_cols),
        "n_preprocessed_features": int(X_train.shape[1]),
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_cols": TARGET_COLS,
        "device": str(device),
        "best_val_loss": best_val,
        "elapsed_seconds": elapsed,
    })

    save_experiment_outputs(experiment_name, metrics, prediction_out, history, config_out, model if save_model else None)

    display(metrics.sort_values("r2", ascending=False))
    return metrics, prediction_out, history, config_out


def run_residual_experiment(config: dict, seed=42, save_model=True):
    experiment_name = config["experiment_name"]
    print(f"\nRunning experiment: {experiment_name}")

    df = load_modeling_table()
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)
    feature_cols = get_feature_columns(df)

    train_df, val_df, test_df = make_splits(df, seed=seed)

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(train_df, feature_cols)

    X_train = preprocessor.fit_transform(prepare_feature_frame(train_df, numeric_cols, categorical_cols))
    X_val = preprocessor.transform(prepare_feature_frame(val_df, numeric_cols, categorical_cols))
    X_test = preprocessor.transform(prepare_feature_frame(test_df, numeric_cols, categorical_cols))

    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(train_df[TARGET_COLS].values)
    y_val = y_scaler.transform(val_df[TARGET_COLS].values)
    y_test_orig = test_df[TARGET_COLS].values

    model = ResidualMLP(
        input_dim=X_train.shape[1],
        output_dim=len(TARGET_COLS),
        hidden_dim=config.get("hidden_dim", 128),
        num_blocks=config.get("num_blocks", 3),
        dropout=config.get("dropout", 0.20),
    )

    model, history, best_val, device, elapsed = train_torch_model(
        model=model,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        epochs=config.get("epochs", 120),
        batch_size=config.get("batch_size", 64),
        learning_rate=config.get("learning_rate", 1e-3),
        weight_decay=config.get("weight_decay", 1e-4),
        patience=config.get("patience", 20),
        seed=seed,
    )

    model.eval()
    with torch.no_grad():
        pred_scaled = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

    pred = y_scaler.inverse_transform(pred_scaled)
    metrics = evaluate_predictions(y_test_orig, pred, TARGET_COLS)
    metrics["model_family"] = config.get("model_family", "residual_mlp")
    metrics["best_val_loss"] = best_val
    metrics["elapsed_seconds"] = elapsed

    prediction_out = test_df[["case_id", "project_id"] if "project_id" in test_df.columns else ["case_id"]].copy()
    for i, target in enumerate(TARGET_COLS):
        prediction_out[f"{target}_true"] = y_test_orig[:, i]
        prediction_out[f"{target}_pred"] = pred[:, i]

    config_out = dict(config)
    config_out.update({
        "n_rows": len(df),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        "n_raw_features": len(feature_cols),
        "n_preprocessed_features": int(X_train.shape[1]),
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "target_cols": TARGET_COLS,
        "device": str(device),
        "best_val_loss": best_val,
        "elapsed_seconds": elapsed,
    })

    save_experiment_outputs(experiment_name, metrics, prediction_out, history, config_out, model if save_model else None)

    display(metrics.sort_values("r2", ascending=False))
    return metrics, prediction_out, history, config_out


def build_modality_preprocessors(train_df: pd.DataFrame, modalities: dict[str, list[str]]):
    preprocessors = {}
    modality_columns = {}

    for modality, cols in modalities.items():
        cols = [c for c in cols if c in train_df.columns]
        if not cols:
            preprocessors[modality] = None
            modality_columns[modality] = {"numeric": [], "categorical": []}
            continue

        preprocessor, numeric_cols, categorical_cols = build_preprocessor(train_df, cols)
        preprocessors[modality] = preprocessor
        modality_columns[modality] = {"numeric": numeric_cols, "categorical": categorical_cols}

    return preprocessors, modality_columns


def transform_modality(df: pd.DataFrame, preprocessor, columns_info: dict) -> np.ndarray:
    numeric_cols = columns_info["numeric"]
    categorical_cols = columns_info["categorical"]
    if preprocessor is None:
        return np.zeros((len(df), 0), dtype=np.float32)
    return preprocessor.transform(prepare_feature_frame(df, numeric_cols, categorical_cols))


def fit_transform_modality(df: pd.DataFrame, preprocessor, columns_info: dict) -> np.ndarray:
    numeric_cols = columns_info["numeric"]
    categorical_cols = columns_info["categorical"]
    if preprocessor is None:
        return np.zeros((len(df), 0), dtype=np.float32)
    return preprocessor.fit_transform(prepare_feature_frame(df, numeric_cols, categorical_cols))


def run_multibranch_experiment(config: dict, seed=42, save_model=True):
    experiment_name = config["experiment_name"]
    print(f"\nRunning experiment: {experiment_name}")

    df = load_modeling_table()
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)
    feature_cols = get_feature_columns(df)
    modalities = split_feature_modalities(feature_cols)

    print("Modality columns:")
    for modality, cols in modalities.items():
        print(f"  {modality}: {len(cols)}")

    train_df, val_df, test_df = make_splits(df, seed=seed)
    preprocessors, modality_columns = build_modality_preprocessors(train_df, modalities)

    Xc_train = fit_transform_modality(train_df, preprocessors["clinical"], modality_columns["clinical"])
    Xm_train = fit_transform_modality(train_df, preprocessors["mutation"], modality_columns["mutation"])
    Xn_train = fit_transform_modality(train_df, preprocessors["copy_number"], modality_columns["copy_number"])

    Xc_val = transform_modality(val_df, preprocessors["clinical"], modality_columns["clinical"])
    Xm_val = transform_modality(val_df, preprocessors["mutation"], modality_columns["mutation"])
    Xn_val = transform_modality(val_df, preprocessors["copy_number"], modality_columns["copy_number"])

    Xc_test = transform_modality(test_df, preprocessors["clinical"], modality_columns["clinical"])
    Xm_test = transform_modality(test_df, preprocessors["mutation"], modality_columns["mutation"])
    Xn_test = transform_modality(test_df, preprocessors["copy_number"], modality_columns["copy_number"])

    y_scaler = StandardScaler()
    y_train = y_scaler.fit_transform(train_df[TARGET_COLS].values)
    y_val = y_scaler.transform(val_df[TARGET_COLS].values)
    y_test_orig = test_df[TARGET_COLS].values

    model = MultiBranchMultiTaskNet(
        clinical_dim=Xc_train.shape[1],
        mutation_dim=Xm_train.shape[1],
        copy_number_dim=Xn_train.shape[1],
        output_dim=len(TARGET_COLS),
        branch_hidden_dim=config.get("branch_hidden_dim", 64),
        fusion_hidden_dim=config.get("fusion_hidden_dim", 128),
        dropout=config.get("dropout", 0.20),
    )

    model, history, best_val, device, elapsed = train_multibranch_model(
        model=model,
        train_arrays=(Xc_train, Xm_train, Xn_train, y_train),
        val_arrays=(Xc_val, Xm_val, Xn_val, y_val),
        epochs=config.get("epochs", 120),
        batch_size=config.get("batch_size", 64),
        learning_rate=config.get("learning_rate", 1e-3),
        weight_decay=config.get("weight_decay", 1e-4),
        patience=config.get("patience", 20),
        seed=seed,
    )

    model.eval()
    with torch.no_grad():
        pred_scaled = model(
            torch.tensor(Xc_test, dtype=torch.float32).to(device),
            torch.tensor(Xm_test, dtype=torch.float32).to(device),
            torch.tensor(Xn_test, dtype=torch.float32).to(device),
        ).cpu().numpy()

    pred = y_scaler.inverse_transform(pred_scaled)
    metrics = evaluate_predictions(y_test_orig, pred, TARGET_COLS)
    metrics["model_family"] = config.get("model_family", "multibranch_multitask")
    metrics["best_val_loss"] = best_val
    metrics["elapsed_seconds"] = elapsed

    prediction_out = test_df[["case_id", "project_id"] if "project_id" in test_df.columns else ["case_id"]].copy()
    for i, target in enumerate(TARGET_COLS):
        prediction_out[f"{target}_true"] = y_test_orig[:, i]
        prediction_out[f"{target}_pred"] = pred[:, i]

    config_out = dict(config)
    config_out.update({
        "n_rows": len(df),
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        "n_raw_features": len(feature_cols),
        "clinical_raw_cols": modalities["clinical"],
        "mutation_raw_cols": modalities["mutation"],
        "copy_number_raw_cols": modalities["copy_number"],
        "clinical_preprocessed_dim": int(Xc_train.shape[1]),
        "mutation_preprocessed_dim": int(Xm_train.shape[1]),
        "copy_number_preprocessed_dim": int(Xn_train.shape[1]),
        "target_cols": TARGET_COLS,
        "device": str(device),
        "best_val_loss": best_val,
        "elapsed_seconds": elapsed,
    })

    save_experiment_outputs(experiment_name, metrics, prediction_out, history, config_out, model if save_model else None)

    display(metrics.sort_values("r2", ascending=False))
    return metrics, prediction_out, history, config_out


def aggregate_all_metrics() -> pd.DataFrame:
    files = sorted((RESULTS_DIR / "metrics").glob("*_metrics.csv"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def aggregate_all_histories() -> pd.DataFrame:
    files = sorted((RESULTS_DIR / "histories").glob("*_history.csv"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def summarize_experiments(metrics: pd.DataFrame) -> pd.DataFrame:
    if metrics.empty:
        return pd.DataFrame()
    return (
        metrics
        .groupby(["experiment_name", "model_family"], as_index=False)
        .agg(
            mean_r2=("r2", "mean"),
            mean_mae=("mae", "mean"),
            mean_rmse=("rmse", "mean"),
            best_val_loss=("best_val_loss", "min"),
            elapsed_seconds=("elapsed_seconds", "max"),
        )
        .sort_values("mean_r2", ascending=False)
    )